In [24]:
pip install transformers accelerate torch

Note: you may need to restart the kernel to use updated packages.


In [25]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [26]:
modn = "google/flan-t5-large"
tok = AutoTokenizer.from_pretrained(modn)
mod = AutoModelForSeq2SeqLM.from_pretrained(modn, device_map="auto")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [27]:
import torch
dev='cuda' if torch.cuda.is_available else 'cpu'

In [28]:
mod.device

device(type='cuda', index=0)

In [45]:
def generate_thoughts(model, tokenizer, problem, current_path, k=3, max_new_tokens=150):
    prompt = f"""
    Solve the problem step by step.
    Generate {k} different possible next reasoning steps.

    Problem: {problem}
    Current Reasoning: {current_path}

    Next step:
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_return_sequences=k,
        do_sample=True,          # Required for diverse steps
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )
    # T5 returns ONLY the generated text, not the prompt
    thoughts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return [t.strip() for t in thoughts]

In [32]:
problem = "A train leaves Station A at 60 mph. Another leaves Station B at 80 mph..."
steps = generate_thoughts(mod, tok, q=problem, k=3)

for i, step in enumerate(steps, 1):
    print(f"Thought {i}: {step}\n")

Thought 1: This time frame of 80' means that you do an effective 4 hours = 402 * hp! When it reached their destination to 60 speeds you won.The following 10 miles have already left there? __ 3 points on #1 are due but all 1 can make and slur ups leave 1 less right at 5 right?x2] the total is 3,407**$5 because 5 days for 100,900 will leave them able to carry 500+ a miles off before going any faster since 2,300 - 300 in this cycle time = 80 mile rider speed limit while still taking 100 and 283 days due. There will probably not find even 20%+ 2 or 50 miles and only 1 time left each train leaving the last stops and driving into this place then taking 1,700 away then take 5... > 100! Time = 800 seconds(2250 hours mins and 0,828 sec). If (600km to 403hrs) time from now on out leaves for 3,200/500km is 1,700, the difference should still just enough f1 time of 1.

Thought 2: How can their traffic speed differences between minutes. You cannot figure, the speed variation does make these other 3 

In [46]:
def evaluate_thought(model, tokenizer, problem, thought):
    prompt = f"""
    Evaluate the quality of this reasoning step for solving the problem.
    Respond with ONLY a single integer between 1 and 10.

    Problem: {problem}
    Reasoning Step: {thought}

    Score:
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Deterministic evaluation for consistent scoring
    output_ids = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        repetition_penalty=1.1
    )
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    
    # Robust regex parsing: grabs first standalone 1-10 number
    match = re.search(r'\b([1-9]|10)\b', output_text)
    return int(match.group(1)) if match else 5

In [47]:
def tree_of_thought(model, tokenizer, problem, depth=2, breadth=3):
    # State: (full_path_so_far, cumulative_score)
    states = [(f"Problem: {problem}\nReasoning:", 0.0)]

    for d in range(depth):
        new_states = []
        
        for path, current_score in states:
            thoughts = generate_thoughts(model, tokenizer, problem, path, k=breadth)
            
            for thought in thoughts:
                score = evaluate_thought(model, tokenizer, problem, thought)
                new_path = path + "\n" + thought
                new_states.append((new_path, current_score + score))
                
        # Keep top-k best paths for next depth
        new_states.sort(key=lambda x: x[1], reverse=True)
        states = new_states[:breadth]
        
    # Return the highest-scoring complete reasoning path
    return states[0][0].strip()

In [49]:
import re

In [50]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

problem = "A train leaves Station A at 60 mph. Another leaves Station B at 80 mph toward A. They are 280 miles apart. When will they meet?"

final_path = tree_of_thought(mod, tok, problem, depth=2, breadth=3)
print("✅ Best Reasoning Path:\n", final_path)

✅ Best Reasoning Path:
 Problem: A train leaves Station A at 60 mph. Another leaves Station B at 80 mph toward A. They are 280 miles apart. When will they meet?
Reasoning:
It shouldn only taken 80 mint hours because B uses 1020 mg in passenger time. Since time cannot vary too significantly among people in each carriage since this thing costs only 200 so A could pay as part 2 only 3 minutes but both rides did need about 55 minute shift work total, while leaving from it at least 12+hour to waiting there next 10 more hour causing no travel times with only minor extra effort! Now that all you were given 100000 ppn total as to speed ratio (the two people moving forward quickly on and leaving when someone else has it). the average number for 20-30mmph they get 30/10 or some such b*s%d times in line or about 45/10mpp, and the trains speed
If two passenger took 1/5 extra per leg. Since when someone will have 5mil passengers and travel about 1/3 miles each of carriage need more than 15Mil!


In [51]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load a modern causal instruction model (fits in same VRAM as T5-large)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def generate_thoughts(model, tokenizer, problem, current_path, k=3, max_new_tokens=200):
    messages = [
        {"role": "system", "content": "You are a step-by-step reasoning assistant."},
        {"role": "user", "content": f"Problem: {problem}\nCurrent reasoning: {current_path}\n\nGenerate exactly {k} distinct, logical next steps. Output ONLY the steps, numbered 1 to {k}."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs, max_new_tokens=max_new_tokens, num_return_sequences=k,
        do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.1
    )
    # Slice off prompt tokens to get ONLY generated text
    thoughts = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return [t.strip() for t in thoughts]

def evaluate_thought(model, tokenizer, problem, thought):
    messages = [{"role": "user", "content": f"Problem: {problem}\nReasoning Step: {thought}\n\nScore this step 1-10 for mathematical correctness. Reply with ONLY a number."}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    output_ids = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    match = re.search(r'\b([1-9]|10)\b', text)
    return int(match.group(1)) if match else 5

def tree_of_thought(model, tokenizer, problem, depth=2, breadth=3):
    # State: (full_path, cumulative_score)
    states = [("Problem: " + problem + "\nReasoning:", 0.0)]

    for d in range(depth):
        new_states = []
        for path, current_score in states:
            thoughts = generate_thoughts(model, tokenizer, problem, path, k=breadth)
            for t in thoughts:
                score = evaluate_thought(model, tokenizer, problem, t)
                new_states.append((path + "\n" + t, current_score + score))
                
        new_states.sort(key=lambda x: x[1], reverse=True)
        states = new_states[:breadth]
        
    return states[0][0].strip()

# Run it
problem = "A train leaves Station A at 60 mph. Another leaves Station B at 80 mph toward A. They are 280 miles apart. When will they meet?"
result = tree_of_thought(model, tokenizer, problem, depth=2, breadth=3)
print(result)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Problem: A train leaves Station A at 60 mph. Another leaves Station B at 80 mph toward A. They are 280 miles apart. When will they meet?
Reasoning:
1. Calculate the combined speed of both trains.
2. Determine how long it takes for the trains to cover the distance between them.
3. Use the time calculated in step 2 to find when they will meet.
1. Calculate the combined speed of both trains.
2. Determine how long it takes for the trains to cover the distance between them.
3. Use the time calculated in step 2 to find when they will meet.
